In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow import keras
from tensorflow.keras import layers


1. Data Loading

print("Loading Dataset...")
df = pd.read_csv("cattle_health_data.csv")   # change name if different

print(df.head())
print(df.info())


2. Data Cleaning & Null Handling

# Drop duplicates
df.drop_duplicates(inplace=True)

# Fill null values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

print("Null Values After Cleaning:\n", df.isnull().sum())

3. Exploratory Data Analysis (EDA)


plt.figure(figsize=(8,5))
sns.histplot(df['Milk_Yield_L'], bins=30, kde=True)
plt.title("Milk Yield Distribution")
plt.show()

plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()



4. Feature Engineering

# Target
y = df['Milk_Yield_L']

# Features
X = df.drop(['Milk_Yield_L'], axis=1)

categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['int64','float64']).columns


5. Preprocessing & Normalization

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

X_processed = preprocessor.fit_transform(X)

if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()


6. Train Test Split

# Shuffle data
indices = np.arange(X_processed.shape[0])
np.random.shuffle(indices)

X_processed = X_processed[indices]
y = y.iloc[indices]

# Split ratio
split_ratio = 0.8
split_index = int(len(X_processed) * split_ratio)

# Split
X_train = X_processed[:split_index]
X_test  = X_processed[split_index:]

y_train = y[:split_index]
y_test  = y[split_index:]

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)
